In [2]:
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler, PolynomialFeatures
from sklearn.feature_selection import SelectKBest, f_classif
from imblearn.over_sampling import SMOTE, SMOTENC
from imblearn.combine import SMOTETomek
import warnings
warnings.filterwarnings('ignore')

# ------ 1. LOAD ------
df = pd.read_csv('/kaggle/input/datasets/zenishadevani/dataset/ai4i2020 (1).csv')
df.drop(columns=['UDI', 'Product ID'], inplace=True)

# ------ 2. ENCODE ------
le = LabelEncoder()
df['Type_enc'] = le.fit_transform(df['Type'])
df.drop(columns=['Type'], inplace=True)

# ------ 3. BASE DOMAIN FEATURES ------
df['Temp_diff']     = df['Process temperature [K]'] - df['Air temperature [K]']
df['Power_W']       = df['Torque [Nm]'] * (df['Rotational speed [rpm]'] * 2 * np.pi / 60)
df['Wear_x_Torque'] = df['Tool wear [min]'] * df['Torque [Nm]']
df['Stress_proxy']  = df['Torque [Nm]'] / (df['Rotational speed [rpm]'] + 1e-6)
df['Wear_per_RPM']  = df['Tool wear [min]'] / (df['Rotational speed [rpm]'] + 1e-6)

# Physics-based failure conditions
df['HDF_cond'] = ((df['Temp_diff'] < 8.6) & (df['Rotational speed [rpm]'] < 1380)).astype(int)
df['PWF_cond'] = ((df['Power_W'] < 3500) | (df['Power_W'] > 9000)).astype(int)
df['OSF_cond'] = (df['Wear_x_Torque'] > 11000).astype(int)
df['multi_fault_risk'] = df['HDF_cond'] + df['PWF_cond'] + df['OSF_cond']  # aggregate risk score

# Normalized tool wear (0 to 1)
df['Wear_norm'] = df['Tool wear [min]'] / df['Tool wear [min]'].max()

# Torque deviation from mean (anomaly signal)
df['Torque_dev'] = np.abs(df['Torque [Nm]'] - df['Torque [Nm]'].mean())

# Temperature ratio
df['Temp_ratio'] = df['Process temperature [K]'] / df['Air temperature [K]']

# RPM binned (Low / Medium / High)
df['RPM_bin'] = pd.cut(df['Rotational speed [rpm]'], bins=3, labels=[0, 1, 2]).astype(int)

# ------ 4. POLYNOMIAL FEATURES (degree=2, top numeric only) ------
poly_cols = ['Air temperature [K]', 'Process temperature [K]',
             'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]',
             'Temp_diff', 'Power_W']

poly = PolynomialFeatures(degree=2, interaction_only=False, include_bias=False)
poly_arr = poly.fit_transform(df[poly_cols])
poly_df = pd.DataFrame(poly_arr, columns=poly.get_feature_names_out(poly_cols))

# Remove duplicate base columns (already in df)
poly_df = poly_df.drop(columns=poly_cols, errors='ignore')
df = pd.concat([df.reset_index(drop=True), poly_df.reset_index(drop=True)], axis=1)

print(f"After polynomial features: {df.shape[1]} columns")

# ------ 5. TARGET & DROP FAILURE LABELS ------
TARGET = 'Machine failure'
FAILURE_COLS = ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']
feature_cols = [c for c in df.columns if c not in [TARGET] + FAILURE_COLS]

X = df[feature_cols]
y = df[TARGET]

# ------ 6. FEATURE SELECTION (keep top 40) ------
selector = SelectKBest(f_classif, k=40)
X_selected = selector.fit_transform(X, y)
selected_features = X.columns[selector.get_support()].tolist()
X = pd.DataFrame(X_selected, columns=selected_features)

print(f"Selected top 40 features: {selected_features}")

# ------ 7. SCALING ------
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

# ------ 8. SMOTE (use INSIDE CV loop in real training, shown here for demo) ------
print(f"\nBefore SMOTE: {y.value_counts().to_dict()}")
smote = SMOTE(random_state=42, k_neighbors=5)
X_resampled, y_resampled = smote.fit_resample(X_scaled, y)
print(f"After SMOTE:  {pd.Series(y_resampled).value_counts().to_dict()}")

# SMOTETomek variant (SMOTE + remove noisy Tomek links)
smotetomek = SMOTETomek(random_state=42)
X_st, y_st = smotetomek.fit_resample(X_scaled, y)
print(f"After SMOTETomek: {pd.Series(y_st).value_counts().to_dict()}")

print("\n✅ Feature Engineering V2 complete.")
print(f"Final feature count: {X_resampled.shape[1]}")

After polynomial features: 53 columns
Selected top 40 features: ['Air temperature [K]', 'Torque [Nm]', 'Tool wear [min]', 'Temp_diff', 'Power_W', 'Wear_x_Torque', 'Stress_proxy', 'Wear_per_RPM', 'HDF_cond', 'PWF_cond', 'OSF_cond', 'multi_fault_risk', 'Wear_norm', 'Torque_dev', 'Temp_ratio', 'RPM_bin', 'Air temperature [K]^2', 'Air temperature [K] Process temperature [K]', 'Air temperature [K] Torque [Nm]', 'Air temperature [K] Tool wear [min]', 'Air temperature [K] Temp_diff', 'Air temperature [K] Power_W', 'Process temperature [K] Torque [Nm]', 'Process temperature [K] Tool wear [min]', 'Process temperature [K] Temp_diff', 'Process temperature [K] Power_W', 'Rotational speed [rpm] Torque [Nm]', 'Rotational speed [rpm] Tool wear [min]', 'Rotational speed [rpm] Temp_diff', 'Rotational speed [rpm] Power_W', 'Torque [Nm]^2', 'Torque [Nm] Tool wear [min]', 'Torque [Nm] Temp_diff', 'Torque [Nm] Power_W', 'Tool wear [min]^2', 'Tool wear [min] Temp_diff', 'Tool wear [min] Power_W', 'Temp_diff